In [19]:
import json
import requests
from openai import OpenAI

client = OpenAI()

messages = []

def get_popular_movies():
    url = "https://nomad-movies.nomadcoders.workers.dev/movies"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_details(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_credits(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()

def get_movie_videos(id):
    # get movide videos by id   
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/videos"
    resp = requests.get(url, timeout= 10)
    resp.raise_for_status()
    return resp.json()


TOOLS = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Get a list of currently popular movies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Get detailed information about a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Get cast and crew credits for a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type":"function",
        "name":"get_movie_videos",
        "description": "Get videos that is related with movie by id",
        "parameters": {
            "type":"object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "the movie id"
                },
            },
            "required":["id"],
            "additionalProperties": False
        }
    }
]


def call_tool(tool_name, args):
    if tool_name == "get_popular_movies":
        return get_popular_movies()
    elif tool_name == "get_movie_details":
        return get_movie_details(args["id"])
    elif tool_name == "get_movie_credits":
        return get_movie_credits(args["id"])
    elif tool_name == "get_movie_videos":
        return get_movie_videos(args["id"])
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(user_input):
    messages.append({
        "role": "user",
        "content": user_input,
    })
    response = client.responses.create(
        model="gpt-4o-mini",
        input=messages,
        instructions=("You are a helpful movie assistant. "
        "Use the provided tools whenever the user asks for real movie data. " 
        "If the user asks for popular movies, use get_popular_movies and include each movie id next to the title." 
        "ID is REQUIRED NEXT TO TITLE ! "
        "If the user asks for details, use get_movie_details. " 
        "If the user asks for cast or crew, use get_movie_credits. "
        "If the user asks for trailers or videos, use get_movie_videos. "
        "Answer clearly in Korean."
        ),
        tools=TOOLS,
    )
    
    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        messages.append({
            "role": "assistant",
            "content": response.output_text,
        })
        return response.output_text

    call = function_calls[0]

    tool_name = call.name
    args = json.loads(call.arguments)

    print(f"🔧 Function Call:, {tool_name}, and the ID is {args}")

    tool_result = call_tool(tool_name, args)

    final_response = client.responses.create(
        model="gpt-4o-mini",
        previous_response_id=response.id,
        instructions="If the user asks for popular movies, use get_popular_movies and include each movie id next to the title." 
        "ID is REQUIRED NEXT TO TITLE ! ",
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(tool_result, ensure_ascii=False),
        }],
    )

    messages.append({
        "role": "assistant",
        "content": final_response.output_text,
    })

    return final_response.output_text


In [ ]:

result = run_agent("지금 인기 있는 영화 1개만 알려줘")
print(result)


🔧 Function Call:, get_popular_movies, and the ID is {}
지금 인기 있는 영화 중 하나는 **"Your Heart Will Be Broken" (ID: 1523145)**입니다.  

- **개요**: 고등학생 폴리나가 새로운 학교에서 괴롭힘을 당하며 주동 사악한 학생인 바르스와 거래를 맺고, 그녀가 그의 남자친구인 척 하도록 요구합니다. 이 게임을 통해 두 사람은 진짜 감정을 키워가지만, 가족과 동급생들이 그들을 이별시킬 이유가 있습니다.
- **평점**: 6.9
- **개봉일**: 2026-03-26

![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)


In [21]:

result = run_agent("니가 알려준 영화에대해 좀더 자세히알려줘")
print(result)


🔧 Function Call:, get_movie_details, and the ID is {'id': 1523145}
영화 **"Your Heart Will Be Broken" (ID: 1523145)**에 대한 자세한 정보는 다음과 같습니다:

- **원제**: Твоё сердце будет разбито
- **장르**: 로맨스, 드라마
- **개봉일**: 2026-03-26
- **러닝 타임**: 134분
- **평점**: 6.9 (투표 수: 55)
- **국가**: 러시아
- **언어**: 러시아어

### 줄거리
고등학생 폴리나가 새로운 학교에서 괴롭힘에 시달리는 중, 대장이자 주된 괴롭힘 가해자인 바르스와 거래를 맺게 됩니다. 바르스는 폴리나의 남자친구인 척 하며 그녀를 보호해야 하고, 폴리나 또한 그의 요구를 수용해야 합니다. 이 게임을 통해 두 사람은 진정한 감정을 키워가지만, 가족과 동급생들이 그들을 이별시키려는 이유가 생깁니다.

### 제작 정보
- **제작사**:
  - All Media A Start Company
  - START Studio
  - Sverdlovsk Film Studio
  - Cinema Foundation of Russia
  - Yellow, Black & White
  - Premier Studios

### 포스터
![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)

### 배경 이미지
![Backdrop](https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg)

이 영화는 고등학생의 복잡한 감정과 갈등을 다룬 로맨틱 드라마로, 다양한 이슈를 탐구하며 젊은 세대의 관계를 심도 있게 보여줍니다.


In [22]:

result = run_agent("그럼 그영화에 예고편이나 관련동영상이있니?")
print(result)


🔧 Function Call:, get_movie_videos, and the ID is {'id': 1523145}
죄송하지만, 영화 **"Your Heart Will Be Broken" (ID: 1523145)**에 대한 예고편이나 관련 동영상은 현재 제공되지 않고 있습니다. 다른 영화들이나 정보를 원하신다면 말씀해 주세요!
